<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: Graph-Based Agent Workflow_sol </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

## Student Version: Graph-Based Agent Workflow

Welcome to the student version of the Graph-Based Agent Workflow demo! In this notebook, you will implement key components of a graph-based agent system, following the structure and concepts outlined. Your task is to complete the code sections marked with `### TODO` comments.

Good luck!

# Demo: Graph-Based Agent Workflow

**Purpose:**  
This notebook demonstrates how to design and implement **structured agent workflows** using a graph-based approach — the foundation of modern agent orchestration frameworks like LangGraph.

**Why Graph-Based Workflows?**  
In real production systems, agents are not simple linear chains. They need conditional routing, parallel execution, loops, human-in-the-loop, and error recovery. Graph-based design makes these complex workflows clear, maintainable, and visual.

**Key Concepts Covered:**
- Node-based execution model
- Agent nodes vs Tool nodes
- Conditional routing and workflow control
- - Visual workflow representation (Mermaid + LangGraph)
- When to use graph-based orchestration vs simple chains

## 1. Setup

In [1]:
# Run this first
!pip install -q "langgraph" "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*"  --force-reinstall
from datetime import datetime
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print(" Setup complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8

## 2. Why Graph-Based Agent Workflows?

Traditional linear chains (like ReAct) are simple but limited.

**Real Production Needs:**
- Conditional branching (e.g., if tool fails → retry or escalate)
- Parallel tool execution
- Human-in-the-loop approval
- Memory and state management across steps
- Clear visual representation for debugging and collaboration

Graph-based workflows (nodes + edges) solve these problems elegantly.

#Dummy Documents (Company Reports)

We create multiple documents so the agent can search and retrieve relevant information.

In [2]:
documents = [
    {"id": 1, "title": "Q3 2025 Financial Report - Microsoft", "content": "Microsoft reported strong growth in Azure AI revenue. Copilot adoption increased 40%. SDG alignment: SDG 9 and SDG 13."},
    {"id": 2, "title": "NVIDIA AI Strategy 2025", "content": "NVIDIA dominates AI hardware with CUDA and DGX systems. Data center revenue grew 80%. Focused on SDG 9 (Innovation)."},
    {"id": 3, "title": "Google Gemini Project Update", "content": "Google's Gemini model achieved new benchmarks. Heavy investment in responsible AI. Aligns with SDG 9 and SDG 13."},
    {"id": 4, "title": "Amazon AWS AI Services", "content": "Amazon Bedrock and SageMaker are leading enterprise AI platforms. Strong focus on sustainable cloud infrastructure (SDG 12 and SDG 13)."},
    {"id": 5, "title": "IBM Watsonx Enterprise AI", "content": "IBM focuses on hybrid cloud AI and quantum computing. Strong commitment to ethical AI and SDG 17 (Partnerships)."}
]

print(f"Loaded {len(documents)} company documents for research.")

Loaded 5 company documents for research.


## 3. Node-Based Execution Model

In graph-based agents, we break the system into **nodes**:

- **Agent Node**: Uses LLM to decide next action
- **Tool Node**: Executes a specific tool
- **Router Node**: Decides the next path based on conditions


We define separate nodes for different responsibilities:
- Retriever Node: Searches documents
- Analyzer Node: Summarizes and reasons
- Router Node: Decides next step

In [3]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, Sequence
import operator
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4", temperature=0)

class AgentState(TypedDict):
    messages: Annotated[Sequence[str], operator.add]

def retriever_node(state):
    """Search relevant documents."""
    query = state["messages"][-1]
    relevant = [doc for doc in documents if any(word.lower() in doc["content"].lower() for word in query.lower().split())]
    if relevant:
        result = "\n".join([f"{doc['title']}: {doc['content']}" for doc in relevant])
    else:
        result = "No relevant documents found."
    return {"messages": state["messages"] + [f"Retrieved: {result}"]}

def agent_node(state):
    """Use GPT to reason and generate final answer."""
    last_message = state["messages"][-1]
    prompt = f"""Based on the following retrieved information, give a clear and concise final answer.

    Retrieved information: {last_message}

    Final Answer:"""

    response = llm.invoke(prompt)
    return {"messages": state["messages"] + [response.content]}

def router_node(state):
    """Decide next step."""
    last_message = state["messages"][-1]
    if "Retrieved" in last_message:
        return "agent"
    return END

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 4. Building the Graph Workflow

We define nodes and edges to create a structured workflow.

In [4]:
workflow = StateGraph(AgentState)

workflow.add_node("retriever", retriever_node)
workflow.add_node("agent", agent_node)

workflow.set_entry_point("retriever")
workflow.add_conditional_edges("retriever", router_node)
workflow.add_edge("agent", END)

app = workflow.compile()

print("Graph-Based Agent Workflow compiled")

Graph-Based Agent Workflow compiled


## 5. Visual Workflow Representation (Mermaid)

In production, we always visualize agent workflows using Mermaid for clarity and team collaboration.

In [5]:
from IPython.display import display, Markdown

mermaid = """
graph TD
    Start[Start] --> Retriever[Retriever Node<br>Search Documents]
    Retriever --> Agent[Agent Node<br>LLM Reasoning]
    Agent --> End[End]
"""

display(Markdown(f"```mermaid\n{mermaid}\n```"))

```mermaid

graph TD
    Start[Start] --> Retriever[Retriever Node<br>Search Documents]
    Retriever --> Agent[Agent Node<br>LLM Reasoning]
    Agent --> End[End]

```

## Explanation of the Graph-Based Workflow

This graph-based agent follows a clear node-based execution model, which is widely used in production agent systems (such as LangGraph).

### How the Workflow Works:

1. **Retriever Node**  
   This node searches through the provided documents and returns relevant snippets based on the user's query.  
   It simulates a real document search step that production agents perform before reasoning.

2. **Agent Node (LLM Reasoning)**  
   This node uses the GPT model to analyze the retrieved documents and generate a final, coherent answer.  
   In a real system, this is where the LLM decides whether it has enough information or needs to retrieve more data.

3. **Router Node**  
   This node decides the next step based on the current state.  
   If documents were retrieved, it routes to the Agent Node for final reasoning. Otherwise, it ends the workflow.

### Why This Design is Used in Production:

- **Structured and Visual**: The graph makes the workflow easy to understand, debug, and modify.
- **Modular**: Each node has a single responsibility (search, reason, route).
- **Scalable**: You can easily add more nodes (e.g., Human-in-the-loop, Validation, or Parallel Tool Execution).
- **Maintainable**: Conditional routing and clear state flow make complex agent logic manageable.

This is the foundation of modern agent orchestration frameworks. In real production systems, graphs often include error handling nodes, parallel execution, and dynamic routing based on tool results.

## 6. Running the Graph-Based Workflow

Let's execute a sample workflow.

In [6]:
initial_state = {"messages": ["Tell me about Microsoft's main AI projects and which SDG goals they follow."]}

result = app.invoke(initial_state)

print("=== Final Answer ===")
print(result["messages"][-1])

=== Final Answer ===
In Q3 2025, Microsoft reported strong growth in Azure AI revenue with a 40% increase in Copilot adoption, aligning with SDG 9 and SDG 13. NVIDIA dominated AI hardware with CUDA and DGX systems, and its data center revenue grew by 80%, focusing on SDG 9. Google's Gemini model achieved new benchmarks with heavy investment in responsible AI, aligning with SDG 9 and SDG 13. Amazon's Bedrock and SageMaker led enterprise AI platforms with a strong focus on sustainable cloud infrastructure (SDG 12 and SDG 13). IBM focused on hybrid cloud AI and quantum computing, with a strong commitment to ethical AI and SDG 17.


## Conclusion & Key Takeaways

In this demo, we built a **Graph-Based Agent Workflow** that:

- Uses a **Retriever Node** to search dummy company documents (simulating real document search).
- Uses an **Agent Node** powered by GPT to reason and generate the final answer.
- Uses a **Router Node** to control the flow between nodes.

**Main Benefits of Graph-Based Design:**
- Clear visual representation of the workflow
- Easy to add conditional logic and loops
- Better debugging and monitoring
- Scalable for complex multi-step agent systems


